**`Task`: Here we view the the most frequent comorbidity or cooccurance of diagnosis based on the data that is filtered by  filter criteraia**
- - medicated patients with a primary diagnosis
- - age between 1 - 19 
- - sak year 1995 onwards
- - exclude diagnose: "000", "1000", "1999", "999", "2000", "2999", "3000", "3999", "99"

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

MAIN_DATASET = os.getenv("MAIN_DATASET")
data_dir = DATA_DIR = os.getenv("DATA_DIR")
output_plot = os.getenv("OUTPUT_PLOTS")

In [ ]:
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
load_dotenv()

SRC_PARQUET = MAIN_DATASET
DATA_DIR = DATA_DIR
df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")


def get_total_unique_pasient_and_opphold_with_atckode(df):
    total_p = df["pasient_nr"].nunique().compute()
    total_o = df["pasient_nr"].nunique().compute()
    print(f"Total unique pasient_nr with atckode: {total_p}, and opphold_id: {total_o}")
    df_with_atckode = df[df["atckode"].notnull()]
    total_pasient = df_with_atckode["pasient_nr"].nunique().compute()
    total_opphold = df_with_atckode["opphold_id"].nunique().compute()
    print(f"Total unique pasient_nr with atckode: {total_pasient}")
    print(f"Total unique opphold_id with atckode: {total_opphold}")
    return total_pasient, total_opphold


get_total_unique_pasient_and_opphold_with_atckode_patient, total_opphold = (
    get_total_unique_pasient_and_opphold_with_atckode(df)
)
print(
    f"Total unique pasient_nr with atckode: {get_total_unique_pasient_and_opphold_with_atckode_patient}"
)
print(f"Total unique opphold_id with atckode: {total_opphold}")

In [ ]:
# This is to create the above mentioned comorbidity dataset
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
load_dotenv()

df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")
df["sak_igangdato"] = dd.to_datetime(df["sak_igangdato"], errors="coerce")
df["sak_avsldato"] = dd.to_datetime(df["sak_avsldato"], errors="coerce")
cutoff = "1995-01-01"
df = df[(df["sak_igangdato"] >= cutoff) & (df["sak_avsldato"] >= cutoff)]
df["patient_age"] = dd.to_numeric(df["patient_age"], errors="coerce")
df = df[(df["patient_age"] >= 1) & (df["patient_age"] <= 19)]
df = df[df["diag_akse"].isin(["1", "2", "3"])]
print(
    f"Patients • age 1-19 • Axis 1,2,3 • episodes ≥ 1995: {df["pasient_nr"].nunique().compute()}"
)
print(
    f"Contacts • age 1-19 • Axis 1,2,3 • episodes ≥ 1995: {df["opphold_id"].nunique().compute()}"
)
blacklist = ["000", "1000", "1999", "999", "2000", "2999", "3000", "3999", "99","1","2","3","4","5","9", "X6n", "Z724"] # F710* only from axis 1

df_primary = df.fillna({"diag_hoved": "0"})
df_primary = df_primary[df_primary["diag_hoved"] == "1"]
df_primary = df_primary.dropna(subset=["diag_diagnose", "atckode"])
df_primary = df_primary[~df_primary["diag_diagnose"].isin(blacklist)]
axis1_only_blacklist = ["F710"]
# blacklist only for axis 1
df_primary = df_primary[~((df_primary["diag_akse"] == "1") & (df_primary["diag_diagnose"].isin(axis1_only_blacklist)))]
print(f"Patients • age 1-19, Axis 1,2,3 from ≥ 1995, excluding diagnosis codes: {df_primary["pasient_nr"].nunique().compute()}")
print(f"Contacts • age 1-19, Axis 1,2,3 from ≥ 1995, excluding diagnosis codes: {df_primary["opphold_id"].nunique().compute()}")

final_columns = [
    "pasient_nr",
    "sak_nr",
    "diag_diagnose",
    "diag_akse",
    "diag_hoved",
    "atckode",
    "opphold_id",
    
]
final_dd = df_primary[final_columns]
print(
    f"Axis 1 patients {final_dd[final_dd["diag_akse"] == "1"]["pasient_nr"].nunique().compute()}, Axis 2: {final_dd[final_dd["diag_akse"] == "2"]["pasient_nr"].nunique().compute()}, Axis 3: {final_dd[final_dd["diag_akse"] == "3"]["pasient_nr"].nunique().compute()}"
)
print(
    f"Axis 1 contacts {final_dd[final_dd["diag_akse"] == "1"]["opphold_id"].nunique().compute()}, Axis 2: {final_dd[final_dd["diag_akse"] == "2"]["opphold_id"].nunique().compute()}, Axis 3: {final_dd[final_dd["diag_akse"] == "3"]["opphold_id"].nunique().compute()}"
)
# Display the count of contacts in axis 1 which has diag_diagnose in more then one diag_akse (i.e, comorbidity), and how many in axis 2, and how many in axis 3

output_file = DATA_DIR / "diagnose_opphold" / "comorbidity.parquet"
final_dd = final_dd.repartition(npartitions=1)
final_dd.to_parquet(output_file, engine="pyarrow", write_index=False)

print(f"Data successfully saved to {output_file}")

**Here we check the contact count with coocurance/comorbidity in all `Axis 1, 2, 3`** 
***Below at last step i check these non-comorbid contacts diagnosis and medication mapping(like the 2 non-comorbid contact in axis 3..)***

In [ ]:
data_dir = DATA_DIR
parquet_path = DATA_DIR / "diagnose_opphold" / "comorbidity.parquet"

df = dd.read_parquet(parquet_path, engine="pyarrow")
diag_counts = df.groupby("opphold_id")["diag_diagnose"].nunique().reset_index()
diag_counts = diag_counts.rename(columns={"diag_diagnose": "n_diag"})

multi_stays = diag_counts[diag_counts.n_diag > 1][["opphold_id"]]

for axis in ["1", "2", "3"]:
    # dedupe to one row per stay/patient/episode
    axis_df = df[df["diag_akse"] == axis][
        ["opphold_id", "pasient_nr", "sak_nr"]
    ].drop_duplicates()

    total_contacts = axis_df["opphold_id"].nunique().compute()

    axis_multi = axis_df.merge(multi_stays, on="opphold_id", how="inner")
    multi_contacts = axis_multi["opphold_id"].nunique().compute()
    multi_patients = axis_multi["pasient_nr"].nunique().compute()
    multi_episodes = axis_multi["sak_nr"].nunique().compute()

    print(
        f"Axis {axis}: "
        f"{total_contacts} total contacts, "
        f"{multi_contacts} contacts with >1 diagnosis, "
        f"{multi_patients} unique patients, "
        f"{multi_episodes} unique episodes"
    )

##### Step 2: Comorbidity Analysis in all contacts

**Contact level view `Axis 1` comorbidity: Here we checking contact level cooccurance of diseases. We take those contacts which has more than one diagnosis being given per contact** 

In [ ]:
PARQUET_PATH = DATA_DIR + "/diagnose_opphold/comorbidity.parquet"
OUTPUT_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis1_comorbidity.xlsx"
df = dd.read_parquet(PARQUET_PATH, engine="pyarrow")
df_primary = df.compute()
diag_list = (
    df_primary.loc[lambda d: d["diag_akse"] == "1", "diag_diagnose"]
    .drop_duplicates()
    .tolist()
)
rows = []

for primary_code in diag_list:
    primary_pairs = (
        df_primary[df_primary["diag_akse"] == "1"]
        .loc[
            lambda d: d["diag_diagnose"] == primary_code,
            ["pasient_nr", "sak_nr", "opphold_id"],
        ]
        .drop_duplicates(subset=["pasient_nr", "sak_nr", "opphold_id"])
    )

    com = df_primary[["pasient_nr", "sak_nr", "opphold_id", "diag_diagnose"]].merge(
        primary_pairs, on=["pasient_nr", "sak_nr", "opphold_id"], how="inner"
    )
    com = com[com["diag_diagnose"] != primary_code]
    patient_counts = (
        com.groupby("diag_diagnose")["pasient_nr"].nunique().rename("patient_count")
    )
    contact_counts = (
        com.groupby("diag_diagnose")["opphold_id"].nunique().rename("contact_count")
    )
    merged = pd.concat([patient_counts, contact_counts], axis=1).fillna(0).astype(int)
    merged = merged.sort_values(by="patient_count", ascending=False).reset_index()

    merged.columns = ["comorbidity_code", "patient_count", "contact_count"]
    primary_name = mapped_icd_codes.get(primary_code, "Not Mapped")
    for _, r in merged.iterrows():
        com_code = r["comorbidity_code"]
        cnt_pat = r["patient_count"]
        cnt_con = r["contact_count"]
        com_name = mapped_icd_codes.get(com_code, "Not Mapped")

        rows.append(
            {
                "primary_code": primary_code,
                "primary_name": primary_name,
                "comorbidity_code": com_code,
                "comorbidity_name": com_name,
                "patient_count": cnt_pat,
                "contact_count": cnt_con,
            }
        )

results_df = pd.DataFrame(rows)
results_df["rank"] = (
    results_df.groupby("primary_code")["patient_count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="xlsxwriter") as writer:
    results_df.to_excel(writer, sheet_name="Long_Form", index=False)
    wide_df = results_df.pivot_table(
        index=["primary_code", "primary_name"],
        columns="rank",
        values=[
            "comorbidity_code",
            "comorbidity_name",
            "patient_count",
            "contact_count",
        ],
        aggfunc="first",
    )
    wide_df.columns = [
        f"Rank#{rank}_{field}" for field, rank in wide_df.columns.tolist()
    ]
    wide_df = wide_df.reset_index()
    wide_df.to_excel(writer, sheet_name="Wide_Form", index=False)

print(f"Axis 1 comorbidity results saved to Excel →  {OUTPUT_EXCEL}")

**We diaplay the comorbidity into categories and the their prevalance in percentage based on the total comorbid contact and patient ratio `axis 1`**
- TOTAL_AXIS1_COMORBID_CONTACTS = 866
- TOTAL_AXIS1_COMORBID_PATIENT = 584

In [ ]:
AXIS1_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis1_comorbidity.xlsx"
OUTPUT_SUMMARY = (
    os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis1_by_category_summary.xlsx"
)
TOTAL_AXIS1_CONTACTS = 866
df_long = pd.read_excel(AXIS1_EXCEL, sheet_name="Long_Form")

categories = {
    # Axis 1
    "Axis1:F03–F09\nOrganic, symptomatic": lambda c: c.startswith("F0")
    and "F03" <= c.replace(".", "")[:3] <= "F09",
    "Axis1:F10–F19\nPsychoactive substances": lambda c: "F10"
    <= c.replace(".", "")[:3]
    <= "F19",
    "Axis1:F20–F29\nSchizophrenia & paranoias": lambda c: "F20"
    <= c.replace(".", "")[:3]
    <= "F29",
    "Axis1:F30–F39\nAffective/mood disorders": lambda c: "F30"
    <= c.replace(".", "")[:3]
    <= "F39",
    "Axis1:F40–F48\nNeurotic & somatoform": lambda c: "F40"
    <= c.replace(".", "")[:3]
    <= "F48",
    "Axis1:F50–F59\nBehavioral syndromes": lambda c: "F50"
    <= c.replace(".", "")[:3]
    <= "F59",
    "Axis1:F60–F69\nPersonality disorders": lambda c: "F60"
    <= c.replace(".", "")[:3]
    <= "F69",
    "Axis1:F84\nPervasive developmental": lambda c: c.replace(".", "").startswith(
        "F84"
    ),
    "Axis1:F90–F98\nChildhood emotional disorders": lambda c: "F90"
    <= c.replace(".", "")[:3]
    <= "F98",
    "Axis1:F99\nUnspecified mental disorder": lambda c: c.replace(".", "").startswith(
        "F99"
    ),
    "Axis1:R40–R46\nCognition & emotional signs": lambda c: "R40"
    <= c.replace(".", "")[:3]
    <= "R46",
    "Axis1:Z00–Z71\nHealth services contacts": lambda c: c.startswith("Z")
    and "Z00" <= c.replace(".", "")[:3] <= "Z71",
    "Axis1:X6n\nSelf-harm & external causes": lambda c: c.startswith("X6"),
    "Axis1:F710\nModerate retardation": lambda c: c.replace(".", "")[:4] == "F710",
    # Axis 2
    "Axis2:F80\nSpeech & language": lambda c: c.replace(".", "").startswith("F80"),
    "Axis2:F81\nSchool skills & learning": lambda c: c.replace(".", "").startswith(
        "F81"
    ),
    "Axis2:F82\nMotor skills": lambda c: c.replace(".", "").startswith("F82"),
    "Axis2:F83\nMixed specific skills": lambda c: c.replace(".", "").startswith("F83"),
    "Axis2:F88\nOther psychological dev": lambda c: c.replace(".", "").startswith(
        "F88"
    ),
    "Axis2:F89\nUnspecified psych dev": lambda c: c.replace(".", "").startswith("F89"),
    # Axis 3
    "Axis3:F70\nMild mental retardation": lambda c: c.replace(".", "").startswith(
        "F70"
    ),
    "Axis3:F71\nModerate mental retardation": lambda c: c.replace(".", "").startswith(
        "F71"
    ),
    "Axis3:F79\nUnspecified mental retardation": lambda c: c.replace(
        ".", ""
    ).startswith("F79"),
}


def assign_category(code: str) -> str:
    """
    Return the first matching category name for this ICD code.
    If none match, return 'Other'.
    """
    for cat_name, predicate in categories.items():
        try:
            if predicate(code):
                return cat_name
        except Exception:
            pass
    return "Other"


df_long["primary_category"] = df_long["primary_code"].astype(str).apply(assign_category)
df_long = df_long[df_long["primary_category"].str.startswith("Axis1:")].copy()
df_long["comorbidity_category"] = (
    df_long["comorbidity_code"].astype(str).apply(assign_category)
)

agg = (
    df_long.groupby(["primary_category", "comorbidity_category"])
    .agg(
        total_contacts=("contact_count", "sum"), total_patients=("patient_count", "sum")
    )
    .reset_index()
)

summary_rows = []

for primary_cat, group in agg.groupby("primary_category"):
    grp_sorted = group.sort_values(by="total_contacts", ascending=False)
    row = {"Primary Category": primary_cat}
    for rank, (_, r) in enumerate(grp_sorted.iterrows(), start=1):
        com_cat = r["comorbidity_category"]
        ct = r["total_contacts"]
        pt = r["total_patients"]
        row[f"Rank#{rank}"] = f"{com_cat}\nC: {ct/866*100:.2f}%"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Primary Category")

max_rank = max(
    int(col.replace("Rank#", ""))
    for col in summary_df.columns
    if col.startswith("Rank#")
)
for i in range(1, max_rank + 1):
    col_name = f"Rank#{i}"
    if col_name not in summary_df.columns:
        summary_df[col_name] = ""

summary_df.to_excel(OUTPUT_SUMMARY)

print(f"Axis 1 categories summary saved: {OUTPUT_SUMMARY}")

**Contact level `axis 2` comorbidity-coocurance**

In [ ]:
PARQUET_PATH = DATA_DIR + "/diagnose_opphold/comorbidity.parquet"
OUTPUT_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis2_comorbidity.xlsx"
diag_list = [
    "F810",
    "F813",
    "F83",
    "F801",
    "F82",
    "F819",
    "F802",
    "F809",
    "F812",
    "F811",
    "F800",
    "F808",
    "F818",
    "F80",
    "F89",
    "F88",
    "F81",
]

df = dd.read_parquet(PARQUET_PATH, engine="pyarrow")
df_primary = df.compute()

rows = []

for primary_code in diag_list:
    primary_pairs = (
        df_primary[df_primary["diag_hoved"] == "1"]
        .loc[
            lambda d: d["diag_diagnose"] == primary_code,
            ["pasient_nr", "sak_nr", "opphold_id"],
        ]
        .drop_duplicates(subset=["pasient_nr", "sak_nr", "opphold_id"])
    )

    com = df_primary[["pasient_nr", "sak_nr", "opphold_id", "diag_diagnose"]].merge(
        primary_pairs, on=["pasient_nr", "sak_nr", "opphold_id"], how="inner"
    )

    com = com[com["diag_diagnose"] != primary_code]
    patient_counts = (
        com.groupby("diag_diagnose")["pasient_nr"].nunique().rename("patient_count")
    )
    contact_counts = (
        com.groupby("diag_diagnose")["opphold_id"].nunique().rename("contact_count")
    )
    merged = pd.concat([patient_counts, contact_counts], axis=1).fillna(0).astype(int)
    merged = merged.sort_values(by="patient_count", ascending=False).reset_index()
    merged.columns = ["comorbidity_code", "patient_count", "contact_count"]
    primary_name = mapped_icd_codes.get(primary_code, "Not Mapped")
    for _, r in merged.iterrows():
        com_code = r["comorbidity_code"]
        cnt_pat = r["patient_count"]
        cnt_con = r["contact_count"]
        com_name = mapped_icd_codes.get(com_code, "Not Mapped")

        rows.append(
            {
                "primary_code": primary_code,
                "primary_name": primary_name,
                "comorbidity_code": com_code,
                "comorbidity_name": com_name,
                "patient_count": cnt_pat,
                "contact_count": cnt_con,
            }
        )

results_df = pd.DataFrame(rows)
results_df["rank"] = (
    results_df.groupby("primary_code")["patient_count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="xlsxwriter") as writer:
    results_df.to_excel(writer, sheet_name="Long_Form", index=False)

    wide_df = results_df.pivot_table(
        index=["primary_code", "primary_name"],
        columns="rank",
        values=[
            "comorbidity_code",
            "comorbidity_name",
            "patient_count",
            "contact_count",
        ],
        aggfunc="first",
    )

    wide_df.columns = [
        f"Rank#{rank}_{field}" for field, rank in wide_df.columns.tolist()
    ]
    wide_df = wide_df.reset_index()
    wide_df.to_excel(writer, sheet_name="Wide_Form", index=False)

print(f"Axis 2 comorbidity results saved to Excel →  {OUTPUT_EXCEL}")

**We diaplay the comorbidity into categories and the their prevalance in percentage based on the total comorbid contact and patient ratio `axis 2`**
- TOTAL_AXIS1_COMORBID_CONTACTS = 789
- TOTAL_AXIS1_COMORBID_PATIENT = 529

In [ ]:
AXIS2_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis2_comorbidity.xlsx"
OUTPUT_SUMMARY = (
    os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis2_by_category_summary.xlsx"
)
df_long = pd.read_excel(AXIS2_EXCEL, sheet_name="Long_Form")

categories = {
    # Axis 1
    "Axis1:F03–F09\nOrganic, symptomatic": lambda c: c.startswith("F0")
    and "F03" <= c.replace(".", "")[:3] <= "F09",
    "Axis1:F10–F19\nPsychoactive substances": lambda c: "F10"
    <= c.replace(".", "")[:3]
    <= "F19",
    "Axis1:F20–F29\nSchizophrenia & paranoias": lambda c: "F20"
    <= c.replace(".", "")[:3]
    <= "F29",
    "Axis1:F30–F39\nAffective/mood disorders": lambda c: "F30"
    <= c.replace(".", "")[:3]
    <= "F39",
    "Axis1:F40–F48\nNeurotic & somatoform": lambda c: "F40"
    <= c.replace(".", "")[:3]
    <= "F48",
    "Axis1:F50–F59\nBehavioral syndromes": lambda c: "F50"
    <= c.replace(".", "")[:3]
    <= "F59",
    "Axis1:F60–F69\nPersonality disorders": lambda c: "F60"
    <= c.replace(".", "")[:3]
    <= "F69",
    "Axis1:F84\nPervasive developmental": lambda c: c.replace(".", "").startswith(
        "F84"
    ),
    "Axis1:F90–F98\nChildhood emotional disorders": lambda c: "F90"
    <= c.replace(".", "")[:3]
    <= "F98",
    "Axis1:F99\nUnspecified mental disorder": lambda c: c.replace(".", "").startswith(
        "F99"
    ),
    "Axis1:R40–R46\nCognition & emotional signs": lambda c: "R40"
    <= c.replace(".", "")[:3]
    <= "R46",
    "Axis1:Z00–Z71\nHealth services contacts": lambda c: c.startswith("Z")
    and "Z00" <= c.replace(".", "")[:3] <= "Z71",
    "Axis1:X6n\nSelf-harm & external causes": lambda c: c.startswith("X6"),
    "Axis1:F710\nModerate retardation": lambda c: c.replace(".", "")[:4] == "F710",
    # Axis 2
    "Axis2:F80\nSpeech & language": lambda c: c.replace(".", "").startswith("F80"),
    "Axis2:F81\nSchool skills & learning": lambda c: c.replace(".", "").startswith(
        "F81"
    ),
    "Axis2:F82\nMotor skills": lambda c: c.replace(".", "").startswith("F82"),
    "Axis2:F83\nMixed specific skills": lambda c: c.replace(".", "").startswith("F83"),
    "Axis2:F88\nOther psychological dev": lambda c: c.replace(".", "").startswith(
        "F88"
    ),
    "Axis2:F89\nUnspecified psych dev": lambda c: c.replace(".", "").startswith("F89"),
    # Axis 3
    "Axis3:F70\nMild mental retardation": lambda c: c.replace(".", "").startswith(
        "F70"
    ),
    "Axis3:F71\nModerate mental retardation": lambda c: c.replace(".", "").startswith(
        "F71"
    ),
    "Axis3:F79\nUnspecified mental retardation": lambda c: c.replace(
        ".", ""
    ).startswith("F79"),
}


def assign_category(code: str) -> str:
    """
    Return the first matching category name for this ICD code.
    If none match, return 'Other'.
    """
    for cat_name, predicate in categories.items():
        try:
            if predicate(code):
                return cat_name
        except Exception:
            pass
    return "Other"


df_long["primary_category"] = df_long["primary_code"].astype(str).apply(assign_category)

df_long = df_long[df_long["primary_category"].str.startswith("Axis2:")].copy()

df_long["comorbidity_category"] = (
    df_long["comorbidity_code"].astype(str).apply(assign_category)
)

agg = (
    df_long.groupby(["primary_category", "comorbidity_category"])
    .agg(
        total_contacts=("contact_count", "sum"), total_patients=("patient_count", "sum")
    )
    .reset_index()
)


summary_rows = []

for primary_cat, group in agg.groupby("primary_category"):

    grp_sorted = group.sort_values(by="total_contacts", ascending=False)
    row = {"Primary Category": primary_cat}
    for rank, (_, r) in enumerate(grp_sorted.iterrows(), start=1):
        com_cat = r["comorbidity_category"]
        ct = r["total_contacts"]
        pt = r["total_patients"]
        row[f"Rank#{rank}"] = f"{com_cat}\nC: {ct/789*100:.2f}%"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Primary Category")

max_rank = max(
    int(col.replace("Rank#", ""))
    for col in summary_df.columns
    if col.startswith("Rank#")
)
for i in range(1, max_rank + 1):
    col_name = f"Rank#{i}"
    if col_name not in summary_df.columns:
        summary_df[col_name] = ""

summary_df.to_excel(OUTPUT_SUMMARY)

print(f"Axis 2 categories summary saved to {OUTPUT_SUMMARY}")

**Save Contact level `axis 3` as excel file**

In [ ]:
PARQUET_PATH = DATA_DIR + "/diagnose_opphold/comorbidity.parquet"
OUTPUT_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis3_comorbidity.xlsx"
ALPHA = 0.7
WRAP_WIDTH = 25

diag_list = [
    "F700",
    "F701",
    "F708",
    "F719",
    "F711",
    "F710",
    "F791",
    "F709",
]


df = dd.read_parquet(PARQUET_PATH, engine="pyarrow")
df_primary = df.compute()

rows = []

for primary_code in diag_list:

    primary_pairs = (
        df_primary[df_primary["diag_hoved"] == "1"]
        .loc[
            lambda d: d["diag_diagnose"] == primary_code,
            ["pasient_nr", "sak_nr", "opphold_id"],
        ]
        .drop_duplicates(subset=["pasient_nr", "sak_nr", "opphold_id"])
    )

    com = df_primary[["pasient_nr", "sak_nr", "opphold_id", "diag_diagnose"]].merge(
        primary_pairs, on=["pasient_nr", "sak_nr", "opphold_id"], how="inner"
    )

    com = com[com["diag_diagnose"] != primary_code]

    patient_counts = (
        com.groupby("diag_diagnose")["pasient_nr"].nunique().rename("patient_count")
    )
    contact_counts = (
        com.groupby("diag_diagnose")["opphold_id"].nunique().rename("contact_count")
    )

    merged = pd.concat([patient_counts, contact_counts], axis=1).fillna(0).astype(int)
    merged = merged.sort_values(by="patient_count", ascending=False).reset_index()
    merged.columns = ["comorbidity_code", "patient_count", "contact_count"]
    primary_name = mapped_icd_codes.get(primary_code, "Not Mapped")
    for _, r in merged.iterrows():
        com_code = r["comorbidity_code"]
        cnt_pat = r["patient_count"]
        cnt_con = r["contact_count"]
        com_name = mapped_icd_codes.get(com_code, "Not Mapped")

        rows.append(
            {
                "primary_code": primary_code,
                "primary_name": primary_name,
                "comorbidity_code": com_code,
                "comorbidity_name": com_name,
                "patient_count": cnt_pat,
                "contact_count": cnt_con,
            }
        )

results_df = pd.DataFrame(rows)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="xlsxwriter") as writer:

    results_df.to_excel(writer, sheet_name="Long_Form", index=False)

    results_df["rank"] = (
        results_df.groupby("primary_code")["patient_count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    wide_df = results_df.pivot_table(
        index=["primary_code", "primary_name"],
        columns="rank",
        values=[
            "comorbidity_code",
            "comorbidity_name",
            "patient_count",
            "contact_count",
        ],
        aggfunc="first",
    )

    wide_df.columns = [
        f"Rank#{rank}_{field}" for field, rank in wide_df.columns.tolist()
    ]
    wide_df = wide_df.reset_index()
    wide_df.to_excel(writer, sheet_name="Wide_Form", index=False)


print(f"All comorbidity results saved to Excel →  {OUTPUT_EXCEL}")

In [ ]:
AXIS3_EXCEL = os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis3_comorbidity.xlsx"
OUTPUT_SUMMARY = (
    os.getenv("AXES_COMORBID_EXCEL_PATH") + "axis3_by_category_summary.xlsx"
)
df_long = pd.read_excel(AXIS3_EXCEL, sheet_name="Long_Form")

categories = {
    # Axis 1
    "Axis1:F03–F09\nOrganic, symptomatic": lambda c: c.startswith("F0")
    and "F03" <= c.replace(".", "")[:3] <= "F09",
    "Axis1:F10–F19\nPsychoactive substances": lambda c: "F10"
    <= c.replace(".", "")[:3]
    <= "F19",
    "Axis1:F20–F29\nSchizophrenia & paranoias": lambda c: "F20"
    <= c.replace(".", "")[:3]
    <= "F29",
    "Axis1:F30–F39\nAffective/mood disorders": lambda c: "F30"
    <= c.replace(".", "")[:3]
    <= "F39",
    "Axis1:F40–F48\nNeurotic & somatoform": lambda c: "F40"
    <= c.replace(".", "")[:3]
    <= "F48",
    "Axis1:F50–F59\nBehavioral syndromes": lambda c: "F50"
    <= c.replace(".", "")[:3]
    <= "F59",
    "Axis1:F60–F69\nPersonality disorders": lambda c: "F60"
    <= c.replace(".", "")[:3]
    <= "F69",
    "Axis1:F84\nPervasive developmental": lambda c: c.replace(".", "").startswith(
        "F84"
    ),
    "Axis1:F90–F98\nChildhood emotional disorders": lambda c: "F90"
    <= c.replace(".", "")[:3]
    <= "F98",
    "Axis1:F99\nUnspecified mental disorder": lambda c: c.replace(".", "").startswith(
        "F99"
    ),
    "Axis1:R40–R46\nCognition & emotional signs": lambda c: "R40"
    <= c.replace(".", "")[:3]
    <= "R46",
    "Axis1:Z00–Z71\nHealth services contacts": lambda c: c.startswith("Z")
    and "Z00" <= c.replace(".", "")[:3] <= "Z71",
    "Axis1:X6n\nSelf-harm & external causes": lambda c: c.startswith("X6"),
    "Axis1:F710\nModerate retardation": lambda c: c.replace(".", "")[:4] == "F710",
    # Axis 2
    "Axis2:F80\nSpeech & language": lambda c: c.replace(".", "").startswith("F80"),
    "Axis2:F81\nSchool skills & learning": lambda c: c.replace(".", "").startswith(
        "F81"
    ),
    "Axis2:F82\nMotor skills": lambda c: c.replace(".", "").startswith("F82"),
    "Axis2:F83\nMixed specific skills": lambda c: c.replace(".", "").startswith("F83"),
    "Axis2:F88\nOther psychological dev": lambda c: c.replace(".", "").startswith(
        "F88"
    ),
    "Axis2:F89\nUnspecified psych dev": lambda c: c.replace(".", "").startswith("F89"),
    # Axis 3
    "Axis3:F70\nMild mental retardation": lambda c: c.replace(".", "").startswith(
        "F70"
    ),
    "Axis3:F71\nModerate mental retardation": lambda c: c.replace(".", "").startswith(
        "F71"
    ),
    "Axis3:F79\nUnspecified mental retardation": lambda c: c.replace(
        ".", ""
    ).startswith("F79"),
}


def assign_category(code: str) -> str:
    """
    Return the first matching category name for this ICD code.
    If none match, return 'Other'.
    """
    for cat_name, predicate in categories.items():
        try:
            if predicate(code):
                return cat_name
        except Exception:
            pass
    return "Other"


df_long["primary_category"] = df_long["primary_code"].astype(str).apply(assign_category)

df_long = df_long[df_long["primary_category"].str.startswith("Axis3:")].copy()

df_long["comorbidity_category"] = (
    df_long["comorbidity_code"].astype(str).apply(assign_category)
)

agg = (
    df_long.groupby(["primary_category", "comorbidity_category"])
    .agg(
        total_contacts=("contact_count", "sum"), total_patients=("patient_count", "sum")
    )
    .reset_index()
)

summary_rows = []

for primary_cat, group in agg.groupby("primary_category"):

    grp_sorted = group.sort_values(by="total_contacts", ascending=False)
    row = {"Primary Category": primary_cat}
    for rank, (_, r) in enumerate(grp_sorted.iterrows(), start=1):
        com_cat = r["comorbidity_category"]
        ct = r["total_contacts"]
        pt = r["total_patients"]
        row[f"Rank#{rank}"] = f"{com_cat}\nC: {ct/63*100:.2f}%"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Primary Category")

max_rank = max(
    int(col.replace("Rank#", ""))
    for col in summary_df.columns
    if col.startswith("Rank#")
)
for i in range(1, max_rank + 1):
    col_name = f"Rank#{i}"
    if col_name not in summary_df.columns:
        summary_df[col_name] = ""

summary_df.to_excel(OUTPUT_SUMMARY)

print(f"Axis 3 categories summary saved to {OUTPUT_SUMMARY}")

##### Step 3: Mapping the medication and diagnosis in all non comorbid contacts in respective axes

**Last step: To view the mapping of medication and diagnosis for `non-comorbid` contacts**

***Below is an end-to-end recipe to pull out just the two `axis-3` contacts with exactly one diagnosis, list their diagnosis + medication, bucket them into your F70/F71/F79 groups, and then build & write the exact same “DiagnosisCategory × MedicationCategory” pivot you did before—but now for those two contacts only***

In [ ]:
parquet_path = data_dir / "diagnose_opphold" / "comorbidity.parquet"
json_path = os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json"
output_path = os.getenv("AXES_COMORBID_EXCEL_PATH") + "/axis3/axis3_noncomorbid.xlsx"

with open(json_path, "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}
df = dd.read_parquet(parquet_path, engine="pyarrow")
full_diag_counts = (
    df.groupby("opphold_id")["diag_diagnose"]
    .nunique()
    .rename("n_diag")
    .reset_index()
    .compute()
)
multi_stays = full_diag_counts.loc[full_diag_counts.n_diag > 1, "opphold_id"].tolist()
print(f"Excluding {len(multi_stays)} stays with >1 diagnosis")
axis3_single = (
    df[(df.diag_akse == "3") & (~df.opphold_id.isin(multi_stays))][
        ["opphold_id", "diag_diagnose", "atckode"]
    ]
    .drop_duplicates()
    .compute()
)
print(
    f"Keeping {axis3_single.opphold_id.nunique():d} axis 3 contacts with exactly 1 diagnosis"
)

axis3_single = axis3_single.rename(columns={"atckode": "med_code"})
axis3_single["med_name"] = axis3_single["med_code"].map(
    lambda c: atc_map.get(c, "Unknown")
)
axis3_single["count"] = 1
TOTAL = axis3_single.opphold_id.nunique()
categories = {
    "Axis3:F70\nMild mental retardation": lambda c: c.replace(".", "").startswith(
        "F70"
    ),
    "Axis3:F71\nModerate mental retardation": lambda c: c.replace(".", "").startswith(
        "F71"
    ),
    "Axis3:F79\nUnspecified mental retardation": lambda c: c.replace(
        ".", ""
    ).startswith("F79"),
}


def categorize_psych(med_code: str) -> str:
    m = med_code.upper()
    if m.startswith("N") and len(m) >= 4:
        p3 = m[:4]
        if p3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if p3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if p3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if p3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if p3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if p3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if p3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if m.startswith("N03"):
            return "N03-Antiepileptics nervous system"
    return "Other Non-Psychotropic"


rows = []
for diag_cat, selector in categories.items():
    sub = axis3_single[axis3_single.diag_diagnose.apply(selector)].copy()
    sub["MedicationCategory"] = sub.med_code.apply(categorize_psych)

    agg = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )

    for med_cat, grp in agg.groupby("MedicationCategory"):
        grp = grp.sort_values("TotalContactCount", ascending=False)
        ranks = [
            f"{r.med_code} {r.med_name} ({(r.TotalContactCount/TOTAL)*100:.2f}%)"
            for _, r in grp.iterrows()
        ]
        rec = {"DiagnosisCategory": diag_cat, "MedicationCategory": med_cat}
        for i, txt in enumerate(ranks, start=1):
            rec[f"MedicationRank{i}(%)"] = txt
        rows.append(rec)


all_ranks = sorted(
    {key for rec in rows for key in rec if key.startswith("MedicationRank")},
    key=lambda x: int(x.replace("MedicationRank", "").split("(")[0]),
)

cols = ["DiagnosisCategory", "MedicationCategory"] + all_ranks
result = pd.DataFrame(rows, columns=cols)

result.to_excel(output_path, index=False)
print(f"Saved summary to: {output_path}")

**To pull out just the two `axis-2` contacts with exactly one diagnosis, list their diagnosis + medication, bucket them into categories**

In [ ]:
parquet_path = data_dir / "diagnose_opphold" / "comorbidity.parquet"
json_path = os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json"
output_path = os.getenv("AXES_COMORBID_EXCEL_PATH") + "/axis2/axis2_noncomorbid.xlsx"


with open(json_path, "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}

df = dd.read_parquet(parquet_path, engine="pyarrow")

full_diag_counts = (
    df.groupby("opphold_id")["diag_diagnose"]
    .nunique()
    .rename("n_diag")
    .reset_index()
    .compute()
)
multi_stays = full_diag_counts.loc[full_diag_counts.n_diag > 1, "opphold_id"].tolist()
print(f"Excluding {len(multi_stays)} stays with >1 diagnosis")

axis3_single = (
    df[(df.diag_akse == "2") & (~df.opphold_id.isin(multi_stays))][
        ["opphold_id", "diag_diagnose", "atckode"]
    ]
    .drop_duplicates()
    .compute()
)
print(
    f"Keeping {axis3_single.opphold_id.nunique():d} axis 2 contacts with exactly 1 diagnosis"
)

axis3_single = axis3_single.rename(columns={"atckode": "med_code"})
axis3_single["med_name"] = axis3_single["med_code"].map(
    lambda c: atc_map.get(c, "Unknown")
)
axis3_single["count"] = 1

TOTAL = axis3_single.opphold_id.nunique()
print(TOTAL)

categories = {
    "F80\nSpecific developmental disorders of speech and language": lambda c: c.replace(
        ".", ""
    ).startswith("F80"),
    "F81\nSpecific developmental disorders of school skills, learning disabilities": lambda c: c.replace(
        ".", ""
    ).startswith(
        "F81"
    ),
    "F82\nSpecific developmental disorder in motor skills": lambda c: c.replace(
        ".", ""
    ).startswith("F82"),
    "F83\nMixed developmental disorder in specific skills": lambda c: c.replace(
        ".", ""
    ).startswith("F83"),
    "F88\nOther disorder of psychological development": lambda c: c.replace(
        ".", ""
    ).startswith("F88"),
    "F89\nUnspecified disorder of psychological development": lambda c: c.replace(
        ".", ""
    ).startswith("F89"),
}


def categorize_psych(med_code: str) -> str:
    m = med_code.upper()
    if m.startswith("N") and len(m) >= 4:
        p3 = m[:4]
        if p3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if p3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if p3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if p3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if p3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if p3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if p3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if m.startswith("N03"):
            return "N03-Antiepileptics nervous system"
    return "Other Non-Psychotropic"


rows = []
for diag_cat, selector in categories.items():
    sub = axis3_single[axis3_single.diag_diagnose.apply(selector)].copy()
    sub["MedicationCategory"] = sub.med_code.apply(categorize_psych)

    agg = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )

    for med_cat, grp in agg.groupby("MedicationCategory"):
        grp = grp.sort_values("TotalContactCount", ascending=False)
        ranks = [
            f"{r.med_code} {r.med_name} ({(r.TotalContactCount/TOTAL)*100:.2f}%)"
            for _, r in grp.iterrows()
        ]
        rec = {"DiagnosisCategory": diag_cat, "MedicationCategory": med_cat}
        for i, txt in enumerate(ranks, start=1):
            rec[f"MedicationRank{i}(%)"] = txt
        rows.append(rec)

all_ranks = sorted(
    {key for rec in rows for key in rec if key.startswith("MedicationRank")},
    key=lambda x: int(x.replace("MedicationRank", "").split("(")[0]),
)

cols = ["DiagnosisCategory", "MedicationCategory"] + all_ranks
result = pd.DataFrame(rows, columns=cols)

result.to_excel(output_path, index=False)
print(f"Saved summary to: {output_path}")

**To pull out just the two `axis-1` contacts with exactly one diagnosis, list their diagnosis + medication, bucket them into categories**

In [ ]:
parquet_path = data_dir / "diagnose_opphold" / "comorbidity.parquet"
json_path = os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json"
output_path = os.getenv("AXES_COMORBID_EXCEL_PATH") + "/axis1/axis1_noncomorbid.xlsx"

with open(json_path, "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}

df = dd.read_parquet(parquet_path, engine="pyarrow")

full_diag_counts = (
    df.groupby("opphold_id")["diag_diagnose"]
    .nunique()
    .rename("n_diag")
    .reset_index()
    .compute()
)
multi_stays = full_diag_counts.loc[full_diag_counts.n_diag > 1, "opphold_id"].tolist()
print(f"Excluding {len(multi_stays)} stays with >1 diagnosis")

axis3_single = (
    df[(df.diag_akse == "1") & (~df.opphold_id.isin(multi_stays))][
        ["opphold_id", "diag_diagnose", "atckode"]
    ]
    .drop_duplicates()
    .compute()
)
print(
    f"Keeping {axis3_single.opphold_id.nunique():d} axis 1 contacts with exactly 1 diagnosis"
)

# 6) Map ATC→med_name and add a count column
axis3_single = axis3_single.rename(columns={"atckode": "med_code"})
axis3_single["med_name"] = axis3_single["med_code"].map(
    lambda c: atc_map.get(c, "Unknown")
)
axis3_single["count"] = 1

TOTAL = axis3_single.opphold_id.nunique()
print(TOTAL)
# 7) Define your ICD-10 buckets (axis 2)
categories = {
    "Axis1:F03–F09\nOrganic, symptomatic": lambda c: c.startswith("F0")
    and "F03" <= c.replace(".", "")[:3] <= "F09",
    "Axis1:F10–F19\nPsychoactive substances": lambda c: "F10"
    <= c.replace(".", "")[:3]
    <= "F19",
    "Axis1:F20–F29\nSchizophrenia & paranoias": lambda c: "F20"
    <= c.replace(".", "")[:3]
    <= "F29",
    "Axis1:F30–F39\nAffective/mood disorders": lambda c: "F30"
    <= c.replace(".", "")[:3]
    <= "F39",
    "Axis1:F40–F48\nNeurotic & somatoform": lambda c: "F40"
    <= c.replace(".", "")[:3]
    <= "F48",
    "Axis1:F50–F59\nBehavioral syndromes": lambda c: "F50"
    <= c.replace(".", "")[:3]
    <= "F59",
    "Axis1:F60–F69\nPersonality disorders": lambda c: "F60"
    <= c.replace(".", "")[:3]
    <= "F69",
    "Axis1:F84\nPervasive developmental": lambda c: c.replace(".", "").startswith(
        "F84"
    ),
    "Axis1:F90–F98\nChildhood emotional disorders": lambda c: "F90"
    <= c.replace(".", "")[:3]
    <= "F98",
    "Axis1:F99\nUnspecified mental disorder": lambda c: c.replace(".", "").startswith(
        "F99"
    ),
    "Axis1:R40–R46\nCognition & emotional signs": lambda c: "R40"
    <= c.replace(".", "")[:3]
    <= "R46",
    "Axis1:Z00–Z71\nHealth services contacts": lambda c: c.startswith("Z")
    and "Z00" <= c.replace(".", "")[:3] <= "Z71",
    "Axis1:X6n\nSelf-harm & external causes": lambda c: c.startswith("X6"),
    "Axis1:F710\nModerate retardation": lambda c: c.replace(".", "")[:4] == "F710",
}


# 8) MedicationCategory mapper
def categorize_psych(med_code: str) -> str:
    m = med_code.upper()
    if m.startswith("N") and len(m) >= 4:
        p3 = m[:4]
        if p3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if p3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if p3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if p3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if p3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if p3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if p3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if m.startswith("N03"):
            return "N03-Antiepileptics nervous system"
    return "Other Non-Psychotropic"


# 9) Build summary rows
rows = []
for diag_cat, selector in categories.items():
    sub = axis3_single[axis3_single.diag_diagnose.apply(selector)].copy()
    sub["MedicationCategory"] = sub.med_code.apply(categorize_psych)

    agg = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )

    for med_cat, grp in agg.groupby("MedicationCategory"):
        grp = grp.sort_values("TotalContactCount", ascending=False)
        ranks = [
            f"{r.med_code} {r.med_name} ({(r.TotalContactCount/TOTAL)*100:.2f}%)"
            for _, r in grp.iterrows()
        ]
        rec = {"DiagnosisCategory": diag_cat, "MedicationCategory": med_cat}
        for i, txt in enumerate(ranks, start=1):
            rec[f"MedicationRank{i}(%)"] = txt
        rows.append(rec)

all_ranks = sorted(
    {key for rec in rows for key in rec if key.startswith("MedicationRank")},
    key=lambda x: int(x.replace("MedicationRank", "").split("(")[0]),
)

cols = ["DiagnosisCategory", "MedicationCategory"] + all_ranks
result = pd.DataFrame(rows, columns=cols)

result.to_excel(output_path, index=False)
print(f" Saved summary to: {output_path}")